In [1]:

%run ../real/input/Format.ipynb
import ROOT as root
from array import array
root.gErrorIgnoreLevel = root.kFatal
%jsroot on
root.gStyle.SetOptStat(0)
import ctypes

/home/yoren/.local/lib/python3.10/site-packages/nbformat/__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


Welcome to JupyROOT 6.30/06


Error in <TUnixSystem::FindDynamicLibrary>: input/logo/PHENIXTools/lib/libLogoPainter.so does not exist in /home/yoren/bnl/ROOT/install/lib:.:/home/yoren/bnl/ROOT/install/lib:/lib/x86_64-linux-gnu/glibc-hwcaps/x86-64-v3:/lib/x86_64-linux-gnu/glibc-hwcaps/x86-64-v2:/lib/x86_64-linux-gnu/tls/haswell/x86_64:/lib/x86_64-linux-gnu/tls/haswell:/lib/x86_64-linux-gnu/tls/x86_64:/lib/x86_64-linux-gnu/tls:/lib/x86_64-linux-gnu/haswell/x86_64:/lib/x86_64-linux-gnu/haswell:/lib/x86_64-linux-gnu/x86_64:/lib/x86_64-linux-gnu:/usr/lib/x86_64-linux-gnu/glibc-hwcaps/x86-64-v3:/usr/lib/x86_64-linux-gnu/glibc-hwcaps/x86-64-v2:/usr/lib/x86_64-linux-gnu/tls/haswell/x86_64:/usr/lib/x86_64-linux-gnu/tls/haswell:/usr/lib/x86_64-linux-gnu/tls/x86_64:/usr/lib/x86_64-linux-gnu/tls:/usr/lib/x86_64-linux-gnu/haswell/x86_64:/usr/lib/x86_64-linux-gnu/haswell:/usr/lib/x86_64-linux-gnu/x86_64:/usr/lib/x86_64-linux-gnu:/lib/glibc-hwcaps/x86-64-v3:/lib/glibc-hwcaps/x86-64-v2:/lib/tls/haswell/x86_64:/lib/tls/haswell:/lib

In [2]:
colors=[1,2,4,root.kGreen+2,root.kOrange+4,root.kMagenta,root.kGray,root.kGray,root.kBlack,root.kCyan,root.kOrange,root.kViolet,root.kCyan,root.kGreen,root.kOrange]
file_path="input/"
file_names=["embedding_fastmask5_0.root"]#sim_fastmask_old,embedding_fastmask"m_ee_DCA_535.root"],"m_ee_DCA_536.root"]

In [3]:
pt_bins = [0.8, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 5.0, 6.0, 8.0, 10.0]
pt_bins = [0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0, 2.5, 3.0, 3.5, 4.0, 5.0, 7.0, 10.0]
#pt_bins = [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 5.0, 10.0]

In [4]:
windows = {
    "ee_signal":  (0.01, 0.12),
    "ee_norm":    (0.15, 0.75),
    "eeg_signal": (0.10, 0.16),
    "eeg_norm":   (0.30, 0.80),
    "eeg_fit_1":  (0.05, 0.28),
}

In [5]:

root.gROOT.SetBatch(True)
root.TH1.AddDirectory(False)

configs = [
    "nominal",
    "conv_solution",
    "conv_tight",
    "ecore03",
    "ecore05",
    "pte02",
    "pte04",
    "eid0",
    "eid1",
    "eid3",
    "chi2_4",
    "chi2_5",
    "escale099",
    "escale101",
]

cent_bins = [0, 1, 2, 3, 4]

In [6]:
hist_kinds_sim = [
    "fg2d_eemass_reco",
    "fg2d_eegmass_reco",
]

weight_types = ["w", "unw"]

hist_names_sim = {}

for kind in hist_kinds_sim:
    hist_names_sim[kind] = {}

    for cfg in configs:
        hist_names_sim[kind][cfg] = {}

        for wt in weight_types:
            hist_names_sim[kind][cfg][wt] = {}

            for icent in cent_bins:
                if wt == "w":
                    hname = f"{kind}_{cfg}_sim_icent{icent}"
                else:
                    hname = f"{kind}_{cfg}_sim_unw_icent{icent}"

                hist_names_sim[kind][cfg][wt][icent] = hname

In [7]:
def get_clone(infile, hname, newname=None, verbose=True):
    obj = infile.Get(hname)
    if not obj:
        if verbose:
            print(f"Missing histogram: {hname}")
        return None

    hist = obj.Clone(newname if newname else hname + "_clone")
    hist.SetDirectory(root.nullptr)
    return hist

In [8]:
hists = {}

for ifile, fname in enumerate(file_names):
    full_name = file_path + fname
    infile = root.TFile.Open(full_name, "READ")

    if not infile or infile.IsZombie():
        print(f"Cannot open file: {full_name}")
        continue

    print(f"Reading sim file {ifile}: {full_name}")

    hists[ifile] = {}
    hists[ifile]["filename"] = full_name

    for kind in hist_kinds_sim:
        hists[ifile][kind] = {}

        for cfg in configs:
            hists[ifile][kind][cfg] = {}

            for wt in weight_types:
                hists[ifile][kind][cfg][wt] = {}

                for icent in cent_bins:
                    hname = hist_names_sim[kind][cfg][wt][icent]
                    newname = f"{hname}_file{ifile}"

                    hists[ifile][kind][cfg][wt][icent] = get_clone(
                        infile,
                        hname,
                        newname=newname
                    )

    # true pion hists
    hists[ifile]["truth"] = {}
    for hname in ["h1d_truepiz", "h1d_truepiz_unw"]:
        hists[ifile]["truth"][hname] = get_clone(
            infile,
            hname,
            newname=f"{hname}_file{ifile}",
            verbose=False
        )

    infile.Close()

Reading sim file 0: input/embedding_fastmask5_0.root


In [9]:
ncolls = [  313.0, 129.0, 41.8, 10.1, 2.28]

In [10]:
for icfg in configs:
    for icent in (1,4):
        hists[0]["fg2d_eemass_reco"][icfg]["w"][0].Add(hists[0]["fg2d_eemass_reco"][icfg]["w"][icent],1./ncolls[icent]*ncolls[0])
        hists[0]["fg2d_eemass_reco"][icfg]["unw"][0].Add(hists[0]["fg2d_eemass_reco"][icfg]["unw"][icent],1./ncolls[icent]*ncolls[0])
        hists[0]["fg2d_eegmass_reco"][icfg]["w"][0].Add(hists[0]["fg2d_eegmass_reco"][icfg]["w"][icent],1./ncolls[icent]*ncolls[0])
        hists[0]["fg2d_eegmass_reco"][icfg]["unw"][0].Add(hists[0]["fg2d_eegmass_reco"][icfg]["unw"][icent],1./ncolls[icent]*ncolls[0])

In [11]:
h_fg_ee_nom_cent0 = hists[0]["fg2d_eemass_reco"]["nominal"]["w"][0]
h_bg_ee_nom_cent0 = hists[0]["fg2d_eemass_reco"]["nominal"]["unw"][0]
h_fg_eeg_nom_cent0 = hists[0]["fg2d_eegmass_reco"]["nominal"]["w"][0]
h_bg_eeg_nom_cent0 = hists[0]["fg2d_eegmass_reco"]["nominal"]["unw"][0]

int_windows = {
    "ee": (0.01, 0.12),
    "eeg": (0.09, 0.2),
}

pt_min = 2.0
pt_max = 10.0

m_min_ee_int = 0.15
m_max_ee_int = 0.75

m_min_eeg_int = 0.3
m_max_eeg_int = 0.8

c1 = root.TCanvas("c1", "c1", 1200, 700)
c1.Divide(2, 1)
c1.cd(1)
ee_proj = h_fg_ee_nom_cent0.ProjectionX("px_fg_ee_nom_cent0",h_fg_ee_nom_cent0.GetYaxis().FindBin(pt_min),h_fg_ee_nom_cent0.GetYaxis().FindBin(pt_max))
ee_proj_bg = h_bg_ee_nom_cent0.ProjectionX("px_bg_ee_nom_cent0",h_bg_ee_nom_cent0.GetYaxis().FindBin(pt_min),h_bg_ee_nom_cent0.GetYaxis().FindBin(pt_max)) 
Format_Hist_total(ee_proj, Lcolor=root.kBlue, Mcolor=root.kBlue, title_x="m_{ee} (GeV/c^{2})", title_y="Counts")
ee_proj.GetXaxis().SetRangeUser(0, 0.3)
ee_proj.Draw()
ee_proj_bg.Scale(ee_proj.Integral(ee_proj.FindBin(m_min_ee_int), ee_proj.FindBin(m_max_ee_int))/ee_proj_bg.Integral(ee_proj_bg.FindBin(m_min_ee_int), ee_proj_bg.FindBin(m_max_ee_int)))

Format_Graph(ee_proj_bg, Lcolor=root.kRed, Mcolor=root.kRed)

print(ee_proj.Integral(ee_proj.FindBin(int_windows["ee"][0]), ee_proj.FindBin(int_windows["ee"][1])) - ee_proj_bg.Integral(ee_proj_bg.FindBin(int_windows["ee"][0]), ee_proj_bg.FindBin(int_windows["ee"][1])))

ee_proj_bg.Draw("same")

c1.cd(2)
eeg_proj = h_fg_eeg_nom_cent0.ProjectionX("px_fg_eeg_nom_cent0",h_fg_eeg_nom_cent0.GetYaxis().FindBin(pt_min),h_fg_eeg_nom_cent0.GetYaxis().FindBin(pt_max))
eeg_proj_bg = h_bg_eeg_nom_cent0.ProjectionX("px_bg_eeg_nom_cent0",h_bg_eeg_nom_cent0.GetYaxis().FindBin(pt_min),h_bg_eeg_nom_cent0.GetYaxis().FindBin(pt_max)) 
Format_Hist_total(eeg_proj, Lcolor=root.kBlue, Mcolor=root.kBlue, title_x="m_{ee#gamma} (GeV/c^{2})", title_y="Counts")
eeg_proj.GetXaxis().SetRangeUser(0, 0.3)
eeg_proj.Draw()
eeg_proj_bg.Scale(eeg_proj.Integral(eeg_proj.FindBin(m_min_eeg_int), eeg_proj.FindBin(m_max_eeg_int))/eeg_proj_bg.Integral(eeg_proj_bg.FindBin(m_min_eeg_int), eeg_proj_bg.FindBin(m_max_eeg_int)))
Format_Graph(eeg_proj_bg, Lcolor=root.kRed, Mcolor=root.kRed)
eeg_proj_bg.Draw("same")

print(eeg_proj.Integral(eeg_proj.FindBin(int_windows["eeg"][0]), eeg_proj.FindBin(int_windows["eeg"][1])) - eeg_proj_bg.Integral(eeg_proj_bg.FindBin(int_windows["eeg"][0]), eeg_proj_bg.FindBin(int_windows["eeg"][1])))

c1.Draw()

177556.21128153446
17273.38048099709


QA utilities

In [12]:
import math
import ctypes
import os

ROOT_KEEPALIVE = []

def keep(obj):
    if obj:
        ROOT_KEEPALIVE.append(obj)
    return obj

def check_obj(obj, name):
    if obj is None:
        print(f"[BAD] {name}: None")
        return False
    try:
        cname = obj.ClassName()
        oname = obj.GetName()
    except Exception as e:
        print(f"[BAD] {name}: invalid ROOT object: {e}")
        return False

    print(f"[OK] {name}: {cname}  {oname}")
    return True

def check_hist(h, name):
    if not check_obj(h, name):
        return False
    print(f"     entries = {h.GetEntries():.3g}")
    print(f"     integral = {h.Integral():.3g}")
    print(f"     nbinsX = {h.GetNbinsX()}, nbinsY = {h.GetNbinsY() if hasattr(h, 'GetNbinsY') else '-'}")
    return True

In [13]:
check_hist(hists[0]["fg2d_eemass_reco"]["nominal"]["w"][0], "fg ee nominal cent0")
check_hist(hists[0]["fg2d_eemass_reco"]["nominal"]["unw"][0], "bg ee nominal cent0")
check_hist(hists[0]["fg2d_eegmass_reco"]["nominal"]["unw"][0], "bg eeg nominal cent0")
check_hist(hists[0]["fg2d_eegmass_reco"]["nominal"]["w"][0], "fg eeg nominal cent0")

[OK] fg ee nominal cent0: TH2F  fg2d_eemass_reco_nominal_sim_icent0_file0
     entries = 1.6e+07
     integral = 5.15e+06
     nbinsX = 400, nbinsY = 100
[OK] bg ee nominal cent0: TH2F  fg2d_eemass_reco_nominal_sim_unw_icent0_file0
     entries = 1.6e+07
     integral = 1.59e+07
     nbinsX = 400, nbinsY = 100
[OK] bg eeg nominal cent0: TH2F  fg2d_eegmass_reco_nominal_sim_unw_icent0_file0
     entries = 1.16e+07
     integral = 1.16e+07
     nbinsX = 400, nbinsY = 100
[OK] fg eeg nominal cent0: TH2F  fg2d_eegmass_reco_nominal_sim_icent0_file0
     entries = 1.16e+07
     integral = 1.04e+06
     nbinsX = 400, nbinsY = 100


True

helpers

In [14]:
def clone_hist(h, name):
    if not check_obj(h, f"clone source {name}"):
        return None

    hc = h.Clone(name)
    if not check_obj(hc, f"clone result {name}"):
        return None

    hc.SetDirectory(root.nullptr)
    keep(hc)
    return hc


def project_mass_in_pt(hist2d, pt_min, pt_max, name):
    if not check_obj(hist2d, f"project source {name}"):
        return None

    yaxis = hist2d.GetYaxis()
    y1 = yaxis.FindBin(pt_min + 1e-3)
    y2 = yaxis.FindBin(pt_max - 1e-3)

    print(f"project_mass_in_pt: {name}")
    print(f"  pt range = [{pt_min}, {pt_max})")
    print(f"  y bins   = {y1} to {y2}")

    proj = hist2d.ProjectionX(name, y1, y2)
    if not check_obj(proj, f"projection {name}"):
        return None

    proj.SetDirectory(root.nullptr)
    keep(proj)

    print(f"  proj integral = {proj.Integral():.3g}")
    return proj


def integral_range(hist, xmin, xmax):
    if not check_obj(hist, f"integral source {xmin}_{xmax}"):
        return 0.0, 0.0

    b1 = hist.FindBin(xmin + 1e-3)
    b2 = hist.FindBin(xmax - 1e-3)

    val = 0.0
    err2 = 0.0

    for ib in range(b1, b2 + 1):
        val += hist.GetBinContent(ib)
        err2 += hist.GetBinError(ib)**2

    return float(val), math.sqrt(err2)


def scale_bg_to_sideband(fg, bg, xmin, xmax):
    fg_int, fg_err = integral_range(fg, xmin, xmax)
    bg_int, bg_err = integral_range(bg, xmin, xmax)

    print(f"sideband [{xmin}, {xmax}]: FG={fg_int:.3g}, BG={bg_int:.3g}")

    if bg_int <= 0:
        print("  WARNING: BG sideband <= 0, using scale=1")
        return 1.0

    scale = fg_int / bg_int *0
    print(f"  scale = {scale:.6g}")
    return scale

cnerality sum

In [15]:
def sum_cent_mb(hists, ifile, kind, cfg, name):
    """Sum centralities for one file and one config."""
    hsum = None

    for icent in cent_bins:
        h = hists[ifile][kind][cfg][icent]
        if not h:
            continue

        if hsum is None:
            hsum = clone_hist(h, name)
            hsum.Reset("ICESM")

        hsum.Add(h)

    return hsum


def sum_cent_mb_all_files(hists, kind, cfg, name):
    """Sum centralities and files."""
    hsum = None

    for ifile in sorted(hists.keys()):
        if not isinstance(ifile, int):
            continue

        hmb = sum_cent_mb(hists, ifile, kind, cfg, f"{name}_file{ifile}")
        if not hmb:
            continue

        if hsum is None:
            hsum = clone_hist(hmb, name)
            hsum.Reset("ICESM")

        hsum.Add(hmb)

    return hsum



In [16]:
pt_min = 2.0
pt_max = 10.0

test_ee_fg = project_mass_in_pt(
    hists[0]["fg2d_eemass_reco"]["nominal"]["w"][0],
    pt_min, pt_max,
    "test_ee_fg"
)

test_ee_bg = project_mass_in_pt(
    hists[0]["fg2d_eemass_reco"]["nominal"]["unw"][0],
    pt_min, pt_max,
    "test_ee_bg"
)

[OK] project source test_ee_fg: TH2F  fg2d_eemass_reco_nominal_sim_icent0_file0
project_mass_in_pt: test_ee_fg
  pt range = [2.0, 10.0)
  y bins   = 21 to 100
[OK] projection test_ee_fg: TH1D  test_ee_fg
  proj integral = 5.32e+05
[OK] project source test_ee_bg: TH2F  fg2d_eemass_reco_nominal_sim_unw_icent0_file0
project_mass_in_pt: test_ee_bg
  pt range = [2.0, 10.0)
  y bins   = 21 to 100
[OK] projection test_ee_bg: TH1D  test_ee_bg
  proj integral = 1.38e+07


In [17]:
test_scale_ee = scale_bg_to_sideband(test_ee_fg, test_ee_bg, 0.15, 0.75)
print(test_scale_ee)

[OK] integral source 0.15_0.75: TH1D  test_ee_fg
[OK] integral source 0.15_0.75: TH1D  test_ee_bg
sideband [0.15, 0.75]: FG=1.25e+04, BG=4.85e+05
  scale = 0
0.0


In [18]:
def get_ee_yield(h_fg_2d, h_bg_2d, pt_min, pt_max, cfg):
    h_fg = project_mass_in_pt(
        h_fg_2d,
        pt_min,
        pt_max,
        f"h_ee_fg_{cfg}_pt{pt_min:g}_{pt_max:g}"
    )

    h_bg = project_mass_in_pt(
        h_bg_2d,
        pt_min,
        pt_max,
        f"h_ee_bg_{cfg}_pt{pt_min:g}_{pt_max:g}"
    )

    if h_fg is None or h_bg is None:
        print("get_ee_yield: missing projection")
        return None

    scale = scale_bg_to_sideband(
        h_fg,
        h_bg,
        windows["ee_norm"][0],
        windows["ee_norm"][1]
    )

    h_bg_scaled = clone_hist(
        h_bg,
        f"h_ee_bg_scaled_{cfg}_pt{pt_min:g}_{pt_max:g}"
    )
    h_bg_scaled.Scale(0.0)  # sim/embedding: h_bg_2d is unw/control; do not subtract it

    h_sub = clone_hist(
        h_fg,
        f"h_ee_sub_{cfg}_pt{pt_min:g}_{pt_max:g}"
    )
    h_sub.Add(h_bg_scaled, -1.0)

    y_fg, e_fg = integral_range(h_fg, *windows["ee_signal"])
    y_bg, e_bg = integral_range(h_bg_scaled, *windows["ee_signal"])
    y_sub, e_sub = integral_range(h_sub, *windows["ee_signal"])

    print("EE yield:")
    print(f"  FG signal = {y_fg:.3g}")
    print(f"  BG/control signal (not subtracted) = {y_bg:.3g}")
    print(f"  SUB       = {y_sub:.3g} +/- {e_sub:.3g}")

    return {
        "h_fg": h_fg,
        "h_bg": h_bg_scaled,
        "h_sub": h_sub,
        "scale": 0.0,
        "sideband_scale_diagnostic": scale,
        "yield": y_sub,
        "yield_err": e_sub,
        "fg_int": y_fg,
        "bg_int": y_bg,
    }

In [19]:
ee_info_test = get_ee_yield(
    hists[0]["fg2d_eemass_reco"]["nominal"]["w"][0],
    hists[0]["fg2d_eemass_reco"]["nominal"]["unw"][0],
    2.0,
    10.0,
    "nominal"
)

ee_info_test["yield"]

[OK] project source h_ee_fg_nominal_pt2_10: TH2F  fg2d_eemass_reco_nominal_sim_icent0_file0
project_mass_in_pt: h_ee_fg_nominal_pt2_10
  pt range = [2.0, 10.0)
  y bins   = 21 to 100
[OK] projection h_ee_fg_nominal_pt2_10: TH1D  h_ee_fg_nominal_pt2_10
  proj integral = 5.32e+05
[OK] project source h_ee_bg_nominal_pt2_10: TH2F  fg2d_eemass_reco_nominal_sim_unw_icent0_file0
project_mass_in_pt: h_ee_bg_nominal_pt2_10
  pt range = [2.0, 10.0)
  y bins   = 21 to 100
[OK] projection h_ee_bg_nominal_pt2_10: TH1D  h_ee_bg_nominal_pt2_10
  proj integral = 1.38e+07
[OK] integral source 0.15_0.75: TH1D  h_ee_fg_nominal_pt2_10
[OK] integral source 0.15_0.75: TH1D  h_ee_bg_nominal_pt2_10
sideband [0.15, 0.75]: FG=1.25e+04, BG=4.85e+05
  scale = 0
[OK] clone source h_ee_bg_scaled_nominal_pt2_10: TH1D  h_ee_bg_nominal_pt2_10
[OK] clone result h_ee_bg_scaled_nominal_pt2_10: TH1D  h_ee_bg_scaled_nominal_pt2_10
[OK] clone source h_ee_sub_nominal_pt2_10: TH1D  h_ee_fg_nominal_pt2_10
[OK] clone result h_e

491545.8359967461

In [20]:
c = root.TCanvas("c_ee_test", "c_ee_test", 800, 600)
ee_info_test["h_fg"].GetXaxis().SetRangeUser(0, 0.3)
ee_info_test["h_fg"].SetLineColor(root.kBlue)
ee_info_test["h_fg"].SetMarkerColor(root.kBlue)
ee_info_test["h_fg"].Draw("E")
ee_info_test["h_bg"].SetLineColor(root.kRed)
ee_info_test["h_bg"].SetMarkerColor(root.kRed)
ee_info_test["h_bg"].Draw("same E")
c.Draw()

In [21]:
def get_eeg_subtracted_hist_no_fit(h_fg_2d, h_bg_2d, pt_min, pt_max, cfg):
    h_fg = project_mass_in_pt(
        h_fg_2d,
        pt_min,
        pt_max,
        f"h_eeg_fg_{cfg}_pt{pt_min:g}_{pt_max:g}"
    )

    h_bg = project_mass_in_pt(
        h_bg_2d,
        pt_min,
        pt_max,
        f"h_eeg_bg_{cfg}_pt{pt_min:g}_{pt_max:g}"
    )

    if h_fg is None or h_bg is None:
        print("get_eeg_subtracted_hist_no_fit: missing projection")
        return None

    scale = scale_bg_to_sideband(
        h_fg,
        h_bg,
        windows["eeg_norm"][0],
        windows["eeg_norm"][1]
    )

    h_bg_scaled = clone_hist(
        h_bg,
        f"h_eeg_bg_scaled_{cfg}_pt{pt_min:g}_{pt_max:g}"
    )
    h_bg_scaled.Scale(scale*0)

    h_sub = clone_hist(
        h_fg,
        f"h_eeg_sub_{cfg}_pt{pt_min:g}_{pt_max:g}"
    )
    h_sub.Add(h_bg_scaled, -1.0)

    y_raw, e_raw = integral_range(h_sub, *windows["eeg_signal"])

    print("EEG after mixed-BG subtraction, before fit:")
    print(f"  raw signal window = {y_raw:.3g} +/- {e_raw:.3g}")

    return {
        "h_fg": h_fg,
        "h_bg": h_bg_scaled,
        "h_sub": h_sub,
        "scale": scale,
        "raw_yield": y_raw,
        "raw_yield_err": e_raw,
    }

In [22]:
eeg_sub_test = get_eeg_subtracted_hist_no_fit(
    hists[0]["fg2d_eegmass_reco"]["nominal"]["w"][0],
    hists[0]["fg2d_eegmass_reco"]["nominal"]["unw"][0],
    2.0,
    10.0,
    "nominal"
)

[OK] project source h_eeg_fg_nominal_pt2_10: TH2F  fg2d_eegmass_reco_nominal_sim_icent0_file0
project_mass_in_pt: h_eeg_fg_nominal_pt2_10
  pt range = [2.0, 10.0)
  y bins   = 21 to 100
[OK] projection h_eeg_fg_nominal_pt2_10: TH1D  h_eeg_fg_nominal_pt2_10
  proj integral = 1.45e+05
[OK] project source h_eeg_bg_nominal_pt2_10: TH2F  fg2d_eegmass_reco_nominal_sim_unw_icent0_file0
project_mass_in_pt: h_eeg_bg_nominal_pt2_10
  pt range = [2.0, 10.0)
  y bins   = 21 to 100
[OK] projection h_eeg_bg_nominal_pt2_10: TH1D  h_eeg_bg_nominal_pt2_10
  proj integral = 9.95e+06
[OK] integral source 0.3_0.8: TH1D  h_eeg_fg_nominal_pt2_10
[OK] integral source 0.3_0.8: TH1D  h_eeg_bg_nominal_pt2_10
sideband [0.3, 0.8]: FG=1e+04, BG=6.72e+05
  scale = 0
[OK] clone source h_eeg_bg_scaled_nominal_pt2_10: TH1D  h_eeg_bg_nominal_pt2_10
[OK] clone result h_eeg_bg_scaled_nominal_pt2_10: TH1D  h_eeg_bg_scaled_nominal_pt2_10
[OK] clone source h_eeg_sub_nominal_pt2_10: TH1D  h_eeg_fg_nominal_pt2_10
[OK] clone r

In [23]:
c = root.TCanvas("c_eeg_sub_test", "c_eeg_sub_test", 800, 600)
eeg_sub_test["h_sub"].GetXaxis().SetRangeUser(0, 0.3)
eeg_sub_test["h_sub"].SetLineColor(root.kBlue)
eeg_sub_test["h_sub"].SetMarkerColor(root.kBlue)
eeg_sub_test["h_sub"].Draw("E")
c.Draw()

In [24]:
def safe_root_name(s):
    s = str(s)
    for ch in [".", "-", "+", " ", "/", "[", "]", "(", ")", ","]:
        s = s.replace(ch, "_")
    return s


def fit_one_eeg_hist_test(h_sub, name="test_fit"):
    """
    Minimal safe fit helper.

    For this sim/embedding workflow the fit is diagnostic only:
      - EEG BG is not subtracted.
      - pol2 is multiplied by 0 in pol2_subtracted_yield().
    Therefore, if ROOT/PyROOT gives a bad TF1 pointer after Fit(), return None
    and let pol2_subtracted_yield() use the raw integral.
    """
    if not check_hist(h_sub, f"{name} input"):
        return None

    if h_sub.Integral() <= 0:
        print(f"[BAD] {name}: histogram integral <= 0, skip fit")
        return None

    tag = safe_root_name(f"{name}_{len(ROOT_KEEPALIVE)}")

    amp0 = max(h_sub.GetMaximum(), 1.0)
    mean0 = 0.135
    sigma0 = 0.018

    f = root.TF1(f"f_{tag}", "gaus(0)+pol2(3)", 0.05, 0.28)

    if not check_obj(f, f"TF1 f_{tag}"):
        return None

    root.SetOwnership(f, False)
    keep(f)

    f.SetParameters(amp0, mean0, sigma0, 0.0, 0.0, 0.0)
    f.SetParLimits(1, 0.10, 0.17)
    f.SetParLimits(2, 0.006, 0.050)

    print("Before fit:")
    print("  amp0 =", amp0)
    print("  mean0 =", mean0)
    print("  sigma0 =", sigma0)

    try:
        res = h_sub.Fit(f, "RQS0")
    except Exception as e:
        print(f"[WARNING] ROOT fit failed for {name}: {e}")
        return None

    print("After fit:")

    # In some PyROOT/ROOT5 cases the TF1 proxy becomes invalid after the fit.
    # Do not crash the workflow; fit is not used for subtraction here.
    try:
        status = int(res.Status()) if res else -999
        mean = f.GetParameter(1)
        sigma = f.GetParameter(2)
        chi2 = f.GetChisquare()
        ndf = f.GetNDF()
    except Exception as e:
        print(f"[WARNING] Cannot access fit parameters for {name}: {e}")
        return None

    print("  status =", status)
    print("  mean =", mean)
    print("  sigma =", sigma)
    print("  chi2/ndf =", chi2, "/", ndf)

    return f


In [25]:
f_test = fit_one_eeg_hist_test(eeg_sub_test["h_sub"], "eeg_nominal_pt2_10")

[OK] eeg_nominal_pt2_10 input: TH1D  h_eeg_sub_nominal_pt2_10
     entries = 2.25e+04
     integral = 1.35e+05
     nbinsX = 400, nbinsY = 1
[OK] TF1 f_eeg_nominal_pt2_10_10: TF1  f_eeg_nominal_pt2_10_10
Before fit:
  amp0 = 8809.772450193763
  mean0 = 0.135
  sigma0 = 0.018
After fit:
  status = 0
  mean = 0.13753753114538783
  sigma = 0.015615069203121742
  chi2/ndf = 576.102191869958 / 86


In [26]:
c = root.TCanvas("c_fit_test", "c_fit_test", 800, 600)
eeg_sub_test["h_sub"].GetXaxis().SetRangeUser(0, 0.3)
eeg_sub_test["h_sub"].Draw("E")
f_test.SetLineColor(root.kRed)
f_test.Draw("same")
c.Draw()

In [27]:
def pol2_subtracted_yield(h, f, xmin, xmax):
    if not check_hist(h, "pol2 yield hist"):
        return 0.0, 0.0, 0.0, 0.0

    if f is None or not check_obj(f, "pol2 yield fit"):
        print("[WARNING] No valid fit. Return raw integral without pol2 subtraction.")
        raw, raw_err = integral_range(h, xmin, xmax)
        return raw, raw_err, raw, 0.0

    pol2 = root.TF1(f"pol2_from_{safe_root_name(f.GetName())}_{len(ROOT_KEEPALIVE)}", "pol2", 0.05, 0.28)
    root.SetOwnership(pol2, False)
    keep(pol2)

    pol2.SetParameters(f.GetParameter(3), f.GetParameter(4), f.GetParameter(5))

    b1 = h.FindBin(xmin + 1e-9)
    b2 = h.FindBin(xmax - 1e-9)

    raw = 0.0
    bg = 0.0
    sig = 0.0
    err2 = 0.0

    for ib in range(b1, b2 + 1):
        x = h.GetBinCenter(ib)
        y = h.GetBinContent(ib)
        b = pol2.Eval(x)*0

        raw += y
        bg += b
        sig += y - b
        err2 += h.GetBinError(ib)**2

    err = math.sqrt(err2)

    print("pol2-subtracted yield:")
    print(f"  raw = {raw:.3g}")
    print(f"  pol2 = {bg:.3g}")
    print(f"  signal = {sig:.3g} +/- {err:.3g}")

    return sig, err, raw, bg

In [28]:
y_eeg, e_eeg, raw_eeg, pol2_eeg = pol2_subtracted_yield(
    eeg_sub_test["h_sub"],
    f_test,
    0.09,
    0.20
)

[OK] pol2 yield hist: TH1D  h_eeg_sub_nominal_pt2_10
     entries = 2.25e+04
     integral = 1.35e+05
     nbinsX = 400, nbinsY = 1
[OK] pol2 yield fit: TF1  f_eeg_nominal_pt2_10_10
pol2-subtracted yield:
  raw = 1.17e+05
  pol2 = 0
  signal = 1.17e+05 +/- 788


In [29]:
ratio_test =  ee_info_test["yield"] / y_eeg if y_eeg != 0 else 0.0
ratio_test

4.191842725785452

In [30]:
def one_bin_result(cfg, pt_min, pt_max, ifile=0, icent=0):
    print("="*70)
    print(f"one_bin_result cfg={cfg}, pt=[{pt_min},{pt_max}], file={ifile}, cent={icent}")

    ee_info = get_ee_yield(
        hists[ifile]["fg2d_eemass_reco"][cfg]["w"][icent],
        hists[ifile]["fg2d_eemass_reco"][cfg]["unw"][icent],
        pt_min,
        pt_max,
        cfg
    )

    eeg_pre = get_eeg_subtracted_hist_no_fit(
        hists[ifile]["fg2d_eegmass_reco"][cfg]["w"][icent],
        hists[ifile]["fg2d_eegmass_reco"][cfg]["unw"][icent],
        pt_min,
        pt_max,
        cfg
    )

    f = fit_one_eeg_hist_test(
        eeg_pre["h_sub"],
        f"eeg_{cfg}_pt{pt_min:g}_{pt_max:g}"
    )

    y_eeg, e_eeg, raw_eeg, pol2_eeg = pol2_subtracted_yield(
        eeg_pre["h_sub"],
        f,
        *windows["eeg_signal"]
    )

    ratio = ee_info["yield"] / y_eeg if y_eeg != 0 else 0.0

    print("FINAL ONE BIN")
    print(f"  Yee  = {ee_info['yield']:.6g}")
    print(f"  Yeeg = {y_eeg:.6g}")
    print(f"  ratio = {ratio:.6g}")

    return {
        "ee": ee_info,
        "eeg_pre": eeg_pre,
        "fit": f,
        "y_eeg": y_eeg,
        "e_eeg": e_eeg,
        "ratio": ratio,
    }

In [31]:
res_one = one_bin_result("nominal", 2.0, 10.0, ifile=0, icent=0)

one_bin_result cfg=nominal, pt=[2.0,10.0], file=0, cent=0
[OK] project source h_ee_fg_nominal_pt2_10: TH2F  fg2d_eemass_reco_nominal_sim_icent0_file0
project_mass_in_pt: h_ee_fg_nominal_pt2_10
  pt range = [2.0, 10.0)
  y bins   = 21 to 100
[OK] projection h_ee_fg_nominal_pt2_10: TH1D  h_ee_fg_nominal_pt2_10
  proj integral = 5.32e+05
[OK] project source h_ee_bg_nominal_pt2_10: TH2F  fg2d_eemass_reco_nominal_sim_unw_icent0_file0
project_mass_in_pt: h_ee_bg_nominal_pt2_10
  pt range = [2.0, 10.0)
  y bins   = 21 to 100
[OK] projection h_ee_bg_nominal_pt2_10: TH1D  h_ee_bg_nominal_pt2_10
  proj integral = 1.38e+07
[OK] integral source 0.15_0.75: TH1D  h_ee_fg_nominal_pt2_10
[OK] integral source 0.15_0.75: TH1D  h_ee_bg_nominal_pt2_10
sideband [0.15, 0.75]: FG=1.25e+04, BG=4.85e+05
  scale = 0
[OK] clone source h_ee_bg_scaled_nominal_pt2_10: TH1D  h_ee_bg_nominal_pt2_10
[OK] clone result h_ee_bg_scaled_nominal_pt2_10: TH1D  h_ee_bg_scaled_nominal_pt2_10
[OK] clone source h_ee_sub_nominal_

In [32]:
res_one_low = one_bin_result("nominal", 1.2, 2.0, ifile=0, icent=0)

one_bin_result cfg=nominal, pt=[1.2,2.0], file=0, cent=0
[OK] project source h_ee_fg_nominal_pt1.2_2: TH2F  fg2d_eemass_reco_nominal_sim_icent0_file0
project_mass_in_pt: h_ee_fg_nominal_pt1.2_2
  pt range = [1.2, 2.0)
  y bins   = 13 to 20
[OK] projection h_ee_fg_nominal_pt1.2_2: TH1D  h_ee_fg_nominal_pt1.2_2
  proj integral = 2.15e+06
[OK] project source h_ee_bg_nominal_pt1.2_2: TH2F  fg2d_eemass_reco_nominal_sim_unw_icent0_file0
project_mass_in_pt: h_ee_bg_nominal_pt1.2_2
  pt range = [1.2, 2.0)
  y bins   = 13 to 20
[OK] projection h_ee_bg_nominal_pt1.2_2: TH1D  h_ee_bg_nominal_pt1.2_2
  proj integral = 1.71e+06
[OK] integral source 0.15_0.75: TH1D  h_ee_fg_nominal_pt1.2_2
[OK] integral source 0.15_0.75: TH1D  h_ee_bg_nominal_pt1.2_2
sideband [0.15, 0.75]: FG=5.18e+04, BG=2.09e+04
  scale = 0
[OK] clone source h_ee_bg_scaled_nominal_pt1.2_2: TH1D  h_ee_bg_nominal_pt1.2_2
[OK] clone result h_ee_bg_scaled_nominal_pt1.2_2: TH1D  h_ee_bg_scaled_nominal_pt1.2_2
[OK] clone source h_ee_sub

In [33]:
def sum_hists_list(hist_list, name):
    hsum = None

    for h in hist_list:
        if h is None:
            continue

        if hsum is None:
            hsum = clone_hist(h, name)
            hsum.Reset("ICESM")

        hsum.Add(h)

    if hsum is None:
        print(f"[BAD] sum_hists_list: could not build {name}")
        return None

    print(f"[OK] built {name}: integral={hsum.Integral():.6g}")
    return hsum


def build_mb_one_file(hists, ifile, kind, cfg):
    hs = []
    for icent in cent_bins:
        h = hists[ifile][kind][cfg]["w"][icent]
        hs.append(h)

    return sum_hists_list(hs, f"mb_file{ifile}_{kind}_{cfg}")


def build_mb_all_files(hists, kind, cfg):
    hs = []

    for ifile in sorted(hists.keys()):
        if not isinstance(ifile, int):
            continue

        h_mb_file = build_mb_one_file(hists, ifile, kind, cfg)
        if h_mb_file is not None:
            hs.append(h_mb_file)

    return sum_hists_list(hs, f"mb_all_{kind}_{cfg}")


def build_mb_set(hists, cfg):
    mb = {}

    mb["fg_ee"] = build_mb_all_files(hists, "fg2d_eemass_reco", cfg)
    mb["bg_ee"] = build_mb_all_files(hists, "fg2d_eemass_reco", cfg)
    mb["fg_eeg"] = build_mb_all_files(hists, "fg2d_eegmass_reco", cfg)
    mb["bg_eeg"] = build_mb_all_files(hists, "fg2d_eegmass_reco", cfg)

    return mb

In [34]:
mb_nominal = build_mb_set(hists, "nominal")

check_hist(mb_nominal["fg_ee"], "MB fg ee nominal")
check_hist(mb_nominal["bg_ee"], "MB bg ee nominal")
check_hist(mb_nominal["fg_eeg"], "MB fg eeg nominal")
check_hist(mb_nominal["bg_eeg"], "MB bg eeg nominal")

[OK] clone source mb_file0_fg2d_eemass_reco_nominal: TH2F  fg2d_eemass_reco_nominal_sim_icent0_file0
[OK] clone result mb_file0_fg2d_eemass_reco_nominal: TH2F  mb_file0_fg2d_eemass_reco_nominal
[OK] built mb_file0_fg2d_eemass_reco_nominal: integral=6.39584e+06
[OK] clone source mb_all_fg2d_eemass_reco_nominal: TH2F  mb_file0_fg2d_eemass_reco_nominal
[OK] clone result mb_all_fg2d_eemass_reco_nominal: TH2F  mb_all_fg2d_eemass_reco_nominal
[OK] built mb_all_fg2d_eemass_reco_nominal: integral=6.39584e+06
[OK] clone source mb_file0_fg2d_eemass_reco_nominal: TH2F  fg2d_eemass_reco_nominal_sim_icent0_file0
[OK] clone result mb_file0_fg2d_eemass_reco_nominal: TH2F  mb_file0_fg2d_eemass_reco_nominal
[OK] built mb_file0_fg2d_eemass_reco_nominal: integral=6.39584e+06
[OK] clone source mb_all_fg2d_eemass_reco_nominal: TH2F  mb_file0_fg2d_eemass_reco_nominal
[OK] clone result mb_all_fg2d_eemass_reco_nominal: TH2F  mb_all_fg2d_eemass_reco_nominal
[OK] built mb_all_fg2d_eemass_reco_nominal: integral=

True

In [35]:
pt_min = 2.0
pt_max = 10.0

mb_ee_fg_test = project_mass_in_pt(mb_nominal["fg_ee"], pt_min, pt_max, "mb_ee_fg_test")
mb_ee_bg_test = project_mass_in_pt(mb_nominal["bg_ee"], pt_min, pt_max, "mb_ee_bg_test")

mb_eeg_fg_test = project_mass_in_pt(mb_nominal["fg_eeg"], pt_min, pt_max, "mb_eeg_fg_test")
mb_eeg_bg_test = project_mass_in_pt(mb_nominal["bg_eeg"], pt_min, pt_max, "mb_eeg_bg_test")

[OK] project source mb_ee_fg_test: TH2F  mb_all_fg2d_eemass_reco_nominal
project_mass_in_pt: mb_ee_fg_test
  pt range = [2.0, 10.0)
  y bins   = 21 to 100
[OK] projection mb_ee_fg_test: TH1D  mb_ee_fg_test
  proj integral = 6.64e+05
[OK] project source mb_ee_bg_test: TH2F  mb_all_fg2d_eemass_reco_nominal
project_mass_in_pt: mb_ee_bg_test
  pt range = [2.0, 10.0)
  y bins   = 21 to 100
[OK] projection mb_ee_bg_test: TH1D  mb_ee_bg_test
  proj integral = 6.64e+05
[OK] project source mb_eeg_fg_test: TH2F  mb_all_fg2d_eegmass_reco_nominal
project_mass_in_pt: mb_eeg_fg_test
  pt range = [2.0, 10.0)
  y bins   = 21 to 100
[OK] projection mb_eeg_fg_test: TH1D  mb_eeg_fg_test
  proj integral = 1.8e+05
[OK] project source mb_eeg_bg_test: TH2F  mb_all_fg2d_eegmass_reco_nominal
project_mass_in_pt: mb_eeg_bg_test
  pt range = [2.0, 10.0)
  y bins   = 21 to 100
[OK] projection mb_eeg_bg_test: TH1D  mb_eeg_bg_test
  proj integral = 1.8e+05


In [36]:
c_mb_test = root.TCanvas("c_mb_test", "c_mb_test", 1200, 600)
c_mb_test.Divide(2, 1)

c_mb_test.cd(1)
mb_ee_fg_test.GetXaxis().SetRangeUser(0, 0.3)
mb_ee_fg_test.SetMarkerColor(root.kBlue)
mb_ee_fg_test.SetLineColor(root.kBlue)
mb_ee_fg_test.Draw("E")

ee_scale_mb = scale_bg_to_sideband(mb_ee_fg_test, mb_ee_bg_test, 0.15, 0.75)
mb_ee_bg_draw = clone_hist(mb_ee_bg_test, "mb_ee_bg_draw")
mb_ee_bg_draw.Scale(ee_scale_mb)
mb_ee_bg_draw.SetMarkerColor(root.kRed)
mb_ee_bg_draw.SetLineColor(root.kRed)
mb_ee_bg_draw.Draw("same E")

c_mb_test.cd(2)
mb_eeg_fg_test.GetXaxis().SetRangeUser(0, 0.3)
mb_eeg_fg_test.SetMarkerColor(root.kBlue)
mb_eeg_fg_test.SetLineColor(root.kBlue)
mb_eeg_fg_test.Draw("E")

eeg_scale_mb = scale_bg_to_sideband(mb_eeg_fg_test, mb_eeg_bg_test, 0.30, 0.80)
mb_eeg_bg_draw = clone_hist(mb_eeg_bg_test, "mb_eeg_bg_draw")
mb_eeg_bg_draw.Scale(eeg_scale_mb)
mb_eeg_bg_draw.SetMarkerColor(root.kRed)
mb_eeg_bg_draw.SetLineColor(root.kRed)
mb_eeg_bg_draw.Draw("same E")

c_mb_test.Draw()

[OK] integral source 0.15_0.75: TH1D  mb_ee_fg_test
[OK] integral source 0.15_0.75: TH1D  mb_ee_bg_test
sideband [0.15, 0.75]: FG=1.66e+04, BG=1.66e+04
  scale = 0
[OK] clone source mb_ee_bg_draw: TH1D  mb_ee_bg_test
[OK] clone result mb_ee_bg_draw: TH1D  mb_ee_bg_draw
[OK] integral source 0.3_0.8: TH1D  mb_eeg_fg_test
[OK] integral source 0.3_0.8: TH1D  mb_eeg_bg_test
sideband [0.3, 0.8]: FG=1.22e+04, BG=1.22e+04
  scale = 0
[OK] clone source mb_eeg_bg_draw: TH1D  mb_eeg_bg_test
[OK] clone result mb_eeg_bg_draw: TH1D  mb_eeg_bg_draw


In [37]:
def one_bin_result_mb(cfg, mb, pt_min, pt_max):
    print("="*80)
    print(f"one_bin_result_mb cfg={cfg}, pt=[{pt_min},{pt_max}]")

    ee_info = get_ee_yield(
        mb["fg_ee"],
        mb["bg_ee"],
        pt_min,
        pt_max,
        cfg
    )

    eeg_pre = get_eeg_subtracted_hist_no_fit(
        mb["fg_eeg"],
        mb["bg_eeg"],
        pt_min,
        pt_max,
        cfg
    )

    if ee_info is None or eeg_pre is None:
        print("[BAD] Missing EE or EEG info")
        return None

    f = fit_one_eeg_hist_test(
        eeg_pre["h_sub"],
        f"eeg_mb_{cfg}_pt{pt_min:g}_{pt_max:g}"
    )

    y_eeg, e_eeg, raw_eeg, pol2_eeg = pol2_subtracted_yield(
        eeg_pre["h_sub"],
        f,
        *windows["eeg_signal"]
    )

    y_ee = ee_info["yield"]
    e_ee = ee_info["yield_err"]

    eeg_over_ee = y_eeg / y_ee if y_ee != 0 else float("nan")
    ee_over_eeg = y_ee / y_eeg if y_eeg != 0 else float("nan")

    print("FINAL MB ONE BIN")
    print(f"  Yee        = {y_ee:.6g} +/- {e_ee:.6g}")
    print(f"  Yeeg       = {y_eeg:.6g} +/- {e_eeg:.6g}")
    print(f"  eeg / ee   = {eeg_over_ee:.6g}")
    print(f"  ee / eeg   = {ee_over_eeg:.6g}")

    return {
        "cfg": cfg,
        "pt_min": pt_min,
        "pt_max": pt_max,
        "ee": ee_info,
        "eeg_pre": eeg_pre,
        "fit": f,
        "y_ee": y_ee,
        "e_ee": e_ee,
        "y_eeg": y_eeg,
        "e_eeg": e_eeg,
        "raw_eeg": raw_eeg,
        "pol2_eeg": pol2_eeg,
        "eeg_over_ee": eeg_over_ee,
        "ee_over_eeg": ee_over_eeg,
    }

In [38]:
res_mb_one = one_bin_result_mb("nominal", mb_nominal, 2.0, 10.0)

one_bin_result_mb cfg=nominal, pt=[2.0,10.0]
[OK] project source h_ee_fg_nominal_pt2_10: TH2F  mb_all_fg2d_eemass_reco_nominal
project_mass_in_pt: h_ee_fg_nominal_pt2_10
  pt range = [2.0, 10.0)
  y bins   = 21 to 100
[OK] projection h_ee_fg_nominal_pt2_10: TH1D  h_ee_fg_nominal_pt2_10
  proj integral = 6.64e+05
[OK] project source h_ee_bg_nominal_pt2_10: TH2F  mb_all_fg2d_eemass_reco_nominal
project_mass_in_pt: h_ee_bg_nominal_pt2_10
  pt range = [2.0, 10.0)
  y bins   = 21 to 100
[OK] projection h_ee_bg_nominal_pt2_10: TH1D  h_ee_bg_nominal_pt2_10
  proj integral = 6.64e+05
[OK] integral source 0.15_0.75: TH1D  h_ee_fg_nominal_pt2_10
[OK] integral source 0.15_0.75: TH1D  h_ee_bg_nominal_pt2_10
sideband [0.15, 0.75]: FG=1.66e+04, BG=1.66e+04
  scale = 0
[OK] clone source h_ee_bg_scaled_nominal_pt2_10: TH1D  h_ee_bg_nominal_pt2_10
[OK] clone result h_ee_bg_scaled_nominal_pt2_10: TH1D  h_ee_bg_scaled_nominal_pt2_10
[OK] clone source h_ee_sub_nominal_pt2_10: TH1D  h_ee_fg_nominal_pt2_10


In [39]:
c_mb_fit = root.TCanvas("c_mb_fit", "c_mb_fit", 800, 600)
res_mb_one["eeg_pre"]["h_sub"].GetXaxis().SetRangeUser(0, 0.3)
res_mb_one["eeg_pre"]["h_sub"].Draw("E")
res_mb_one["fit"].SetLineColor(root.kRed)
res_mb_one["fit"].Draw("same")
c_mb_fit.Draw()

In [40]:
fit_out_dir = "output/fits"
os.makedirs(fit_out_dir, exist_ok=True)

def save_one_bin_canvas(res, tag):
    c = root.TCanvas(f"c_{tag}", f"c_{tag}", 1200, 600)
    c.Divide(2, 1)

    c.cd(1)
    h_ee_fg = res["ee"]["h_fg"]
    h_ee_bg = res["ee"]["h_bg"]

    h_ee_fg.GetXaxis().SetRangeUser(0, 0.3)
    h_ee_fg.SetMarkerColor(root.kBlue)
    h_ee_fg.SetLineColor(root.kBlue)
    h_ee_fg.Draw("E")

    h_ee_bg.SetMarkerColor(root.kRed)
    h_ee_bg.SetLineColor(root.kRed)
    h_ee_bg.Draw("same E")

    lat = root.TLatex()
    lat.SetNDC(True)
    lat.SetTextSize(0.035)
    lat.DrawLatex(0.50, 0.82, f"Y_ee = {res['y_ee']:.3g}")

    c.cd(2)
    h_eeg_sub = res["eeg_pre"]["h_sub"]
    fit = res["fit"]

    h_eeg_sub.GetXaxis().SetRangeUser(0, 0.3)
    h_eeg_sub.SetMarkerColor(root.kBlue)
    h_eeg_sub.SetLineColor(root.kBlue)
    h_eeg_sub.Draw("E")

    if fit:
        fit.SetLineColor(root.kRed)
        fit.Draw("same")

    lat.DrawLatex(0.50, 0.82, f"Y_eeg = {res['y_eeg']:.3g}")
    lat.DrawLatex(0.50, 0.77, f"eeg/ee = {res['eeg_over_ee']:.4g}")
    lat.DrawLatex(0.50, 0.72, f"ee/eeg = {res['ee_over_eeg']:.4g}")

    png = f"{fit_out_dir}/{tag}.png"
    pdf = f"{fit_out_dir}/{tag}.pdf"

    c.SaveAs(png)
    c.SaveAs(pdf)

    keep(c)
    print("saved:", png)
    return c

In [41]:
c_saved_test = save_one_bin_canvas(res_mb_one, "test_nominal_mb_pt2_10")
c_saved_test.Draw()

saved: output/fits/test_nominal_mb_pt2_10.png


In [42]:
import pandas as pd

def run_pt_bins_for_config(cfg, mb, save_canvases=False):
    rows = []
    results = {}

    for ipt in range(len(pt_bins) - 1):
        pt_min = pt_bins[ipt]
        pt_max = pt_bins[ipt + 1]-0.005

        print("\n")
        print("#"*80)
        print(f"Running {cfg}: pT [{pt_min}, {pt_max})")

        res = one_bin_result_mb(cfg, mb, pt_min, pt_max)
        if res is None:
            print(f"[WARNING] skipping {cfg} pT [{pt_min},{pt_max}) because result is None")
            continue
        results[(pt_min, pt_max)] = res

        if save_canvases:
            tag = f"{cfg}_mb_pt{str(pt_min).replace('.','p')}_{str(pt_max).replace('.','p')}"
            save_one_bin_canvas(res, tag)

        row = {
            "config": cfg,
            "pt_min": pt_min,
            "pt_max": pt_max,
            "pt_center": 0.5 * (pt_min + pt_max),

            "y_ee": res["y_ee"],
            "e_ee": res["e_ee"],
            "y_eeg": res["y_eeg"],
            "e_eeg": res["e_eeg"],

            "raw_eeg": res["raw_eeg"],
            "pol2_eeg": res["pol2_eeg"],

            "eeg_over_ee": res["eeg_over_ee"],
            "ee_over_eeg": res["ee_over_eeg"],

            "fit_mean": res["fit"].GetParameter(1) if res["fit"] else float("nan"),
            "fit_sigma": abs(res["fit"].GetParameter(2)) if res["fit"] else float("nan"),
            "fit_chi2": res["fit"].GetChisquare() if res["fit"] else float("nan"),
            "fit_ndf": res["fit"].GetNDF() if res["fit"] else -999,
        }

        rows.append(row)

    df = pd.DataFrame(rows)
    return df, results

In [43]:
df_nominal_mb, results_nominal_mb = run_pt_bins_for_config(
    "nominal",
    mb_nominal,
    save_canvases=False
)

df_nominal_mb



################################################################################
Running nominal: pT [0.6, 0.795)
one_bin_result_mb cfg=nominal, pt=[0.6,0.795]
[OK] project source h_ee_fg_nominal_pt0.6_0.795: TH2F  mb_all_fg2d_eemass_reco_nominal
project_mass_in_pt: h_ee_fg_nominal_pt0.6_0.795
  pt range = [0.6, 0.795)
  y bins   = 7 to 8
[OK] projection h_ee_fg_nominal_pt0.6_0.795: TH1D  h_ee_fg_nominal_pt0.6_0.795
  proj integral = 2.74e+05
[OK] project source h_ee_bg_nominal_pt0.6_0.795: TH2F  mb_all_fg2d_eemass_reco_nominal
project_mass_in_pt: h_ee_bg_nominal_pt0.6_0.795
  pt range = [0.6, 0.795)
  y bins   = 7 to 8
[OK] projection h_ee_bg_nominal_pt0.6_0.795: TH1D  h_ee_bg_nominal_pt0.6_0.795
  proj integral = 2.74e+05
[OK] integral source 0.15_0.75: TH1D  h_ee_fg_nominal_pt0.6_0.795
[OK] integral source 0.15_0.75: TH1D  h_ee_bg_nominal_pt0.6_0.795
sideband [0.15, 0.75]: FG=59.6, BG=59.6
  scale = 0
[OK] clone source h_ee_bg_scaled_nominal_pt0.6_0.795: TH1D  h_ee_bg_nominal_pt0.

,config,pt_min,pt_max,pt_center,y_ee,e_ee,y_eeg,e_eeg,raw_eeg,pol2_eeg,eeg_over_ee,ee_over_eeg,fit_mean,fit_sigma,fit_chi2,fit_ndf
0,nominal,0.6,0.795,0.6975,2.735023e+05,21385.169580,53765.114606,6412.171834,53765.114606,0.0,0.196580,5.086984,0.128633,0.008124,197.246822,53
1,nominal,0.8,0.995,0.8975,1.262033e+06,35527.045676,161955.037743,6299.675368,161955.037743,0.0,0.128329,7.792487,0.135019,0.009779,435.650309,86
2,nominal,1.0,1.195,1.0975,1.442338e+06,23408.039344,229347.220703,5096.491679,229347.220703,0.0,0.159011,6.288882,0.134339,0.012693,515.713866,86
3,nominal,1.2,1.395,1.2975,1.047900e+06,13895.137644,172909.202080,3280.953336,172909.202080,0.0,0.165005,6.060405,0.136687,0.012256,486.476898,86
4,nominal,1.4,1.595,1.4975,6.948852e+05,8072.020513,111093.359344,1955.499576,111093.359344,0.0,0.159873,6.254966,0.133760,0.012414,580.909491,86
5,nominal,1.6,1.795,1.6975,4.568333e+05,4782.809263,70814.960938,1100.515219,70814.960938,0.0,0.155013,6.451085,0.136186,0.013047,441.261334,86
6,nominal,1.8,1.995,1.8975,3.191665e+05,3089.077557,56090.102135,865.863402,56090.102135,0.0,0.175739,5.690247,0.137609,0.014155,709.377700,86
7,nominal,2.0,2.495,2.2475,3.946097e+05,2289.307496,78235.240677,696.489943,78235.240677,0.0,0.198260,5.043886,0.136766,0.014390,737.154718,86
8,nominal,2.5,2.995,2.7475,1.375755e+05,751.626418,29970.464336,252.296216,29970.464336,0.0,0.217847,4.590369,0.138108,0.016007,461.599672,86
9,nominal,3.0,3.495,3.2475,4.665921e+04,252.329471,12418.355982,99.274266,12418.355982,0.0,0.266150,3.757277,0.137553,0.018244,318.568955,86


In [44]:
df_nominal_mb.to_csv(f"output/sim_yields_nominal_mb.csv", index=False)

In [45]:
def add_ratio_columns(df):
    df = df.copy()

    # eeg / ee
    df["eeg_over_ee"] = float("nan")
    df["eeg_over_ee_err"] = float("nan")

    # ee / eeg
    df["ee_over_eeg"] = float("nan")
    df["ee_over_eeg_err"] = float("nan")

    for i, row in df.iterrows():
        y_ee = row["y_ee"]
        e_ee = row["e_ee"]
        y_eeg = row["y_eeg"]
        e_eeg = row["e_eeg"]

        if y_ee > 0 and y_eeg > 0:
            r1 = y_eeg / y_ee
            r1_err = r1 * math.sqrt((e_eeg / y_eeg)**2 + (e_ee / y_ee)**2)

            r2 = y_ee / y_eeg
            r2_err = r2 * math.sqrt((e_ee / y_ee)**2 + (e_eeg / y_eeg)**2)

            df.loc[i, "eeg_over_ee"] = r1
            df.loc[i, "eeg_over_ee_err"] = r1_err
            df.loc[i, "ee_over_eeg"] = r2
            df.loc[i, "ee_over_eeg_err"] = r2_err

    return df

In [46]:
df_nominal_mb = add_ratio_columns(df_nominal_mb)
df_nominal_mb

,config,pt_min,pt_max,pt_center,y_ee,e_ee,y_eeg,e_eeg,raw_eeg,pol2_eeg,eeg_over_ee,ee_over_eeg,fit_mean,fit_sigma,fit_chi2,fit_ndf,eeg_over_ee_err,ee_over_eeg_err
0,nominal,0.6,0.795,0.6975,2.735023e+05,21385.169580,53765.114606,6412.171834,53765.114606,0.0,0.196580,5.086984,0.128633,0.008124,197.246822,53,0.028034,0.725449
1,nominal,0.8,0.995,0.8975,1.262033e+06,35527.045676,161955.037743,6299.675368,161955.037743,0.0,0.128329,7.792487,0.135019,0.009779,435.650309,86,0.006162,0.374160
2,nominal,1.0,1.195,1.0975,1.442338e+06,23408.039344,229347.220703,5096.491679,229347.220703,0.0,0.159011,6.288882,0.134339,0.012693,515.713866,86,0.004376,0.173052
3,nominal,1.2,1.395,1.2975,1.047900e+06,13895.137644,172909.202080,3280.953336,172909.202080,0.0,0.165005,6.060405,0.136687,0.012256,486.476898,86,0.003820,0.140293
4,nominal,1.4,1.595,1.4975,6.948852e+05,8072.020513,111093.359344,1955.499576,111093.359344,0.0,0.159873,6.254966,0.133760,0.012414,580.909491,86,0.003372,0.131916
5,nominal,1.6,1.795,1.6975,4.568333e+05,4782.809263,70814.960938,1100.515219,70814.960938,0.0,0.155013,6.451085,0.136186,0.013047,441.261334,86,0.002905,0.120882
6,nominal,1.8,1.995,1.8975,3.191665e+05,3089.077557,56090.102135,865.863402,56090.102135,0.0,0.175739,5.690247,0.137609,0.014155,709.377700,86,0.003202,0.103677
7,nominal,2.0,2.495,2.2475,3.946097e+05,2289.307496,78235.240677,696.489943,78235.240677,0.0,0.198260,5.043886,0.136766,0.014390,737.154718,86,0.002107,0.053596
8,nominal,2.5,2.995,2.7475,1.375755e+05,751.626418,29970.464336,252.296216,29970.464336,0.0,0.217847,4.590369,0.138108,0.016007,461.599672,86,0.002186,0.046067
9,nominal,3.0,3.495,3.2475,4.665921e+04,252.329471,12418.355982,99.274266,12418.355982,0.0,0.266150,3.757277,0.137553,0.018244,318.568955,86,0.002569,0.036264


In [47]:
def make_ratio_graph_from_df(df, ycol, eycol, name, title):
    g = root.TGraphErrors(len(df))
    g.SetName(name)
    g.SetTitle(title)

    for i, row in enumerate(df.itertuples()):
        x = row.pt_center
        ex = 0.5 * (row.pt_max - row.pt_min)
        y = getattr(row, ycol)
        ey = getattr(row, eycol) if eycol in df.columns else 0.0

        g.SetPoint(i, x, y)
        g.SetPointError(i, ex, ey)

    g.SetMarkerStyle(20)
    g.SetMarkerColor(root.kBlue)
    g.SetLineColor(root.kBlue)
    keep(g)
    return g

In [48]:
g_ee = make_ratio_graph_from_df(
    df_nominal_mb,
    "y_ee",
    "e_ee",
    "g_ee_nominal",
    "nominal; p_{T} (GeV/c); ee"
)

g_eeg = make_ratio_graph_from_df(
    df_nominal_mb,
    "y_eeg",
    "e_eeg",
    "g_eeg_nominal",
    "nominal; p_{T} (GeV/c); ee / eeg"
)

c_yield = root.TCanvas("c_yield_nominal", "c_yield_nominal", 1200, 500)
c_yield.Divide(2, 1)

c_yield.cd(1)
g_ee.Draw("AP")

c_yield.cd(2)
g_eeg.Draw("AP")

c_yield.Draw()

c_ratio = root.TCanvas("c_ratio_nominal", "c_ratio_nominal", 800, 600)

g_ee_over_eeg = make_ratio_graph_from_df(
    df_nominal_mb,
    "ee_over_eeg",
    "ee_over_eeg_err",
    "g_ee_over_eeg_nominal",
    "nominal; p_{T} (GeV/c); ee / eeg"
)

c_ratio = root.TCanvas("c_ratio_nominal", "c_ratio_nominal", 600, 500)
g_ee_over_eeg.GetYaxis().SetRangeUser(0, 15)
g_ee_over_eeg.Draw("AP")

c_ratio.Draw()


In [49]:
configs_to_run = [
    "nominal",
    "conv_solution",
    "conv_tight",
    "ecore03",
    "ecore05",
    "pte02",
    "pte04",
    "eid0",
    "eid1",
    "eid3",
    "chi2_4",
    "chi2_5",
    "escale099",
    "escale101",
]

In [50]:
all_dfs = []
all_results = {}

for cfg in configs_to_run:
    print("\n\n")
    print("="*100)
    print(f"BUILDING MB HISTOGRAMS FOR {cfg}")
    print("="*100)

    mb_cfg = build_mb_set(hists, cfg)

    print("\n")
    print("="*100)
    print(f"RUNNING PT BINS FOR {cfg}")
    print("="*100)

    df_cfg, results_cfg = run_pt_bins_for_config(
        cfg,
        mb_cfg,
        save_canvases=False
    )

    all_dfs.append(df_cfg)
    all_results[cfg] = results_cfg

df_all = pd.concat(all_dfs, ignore_index=True)
df_all.to_csv(f"{fit_out_dir}/sim_yields_all_configs_mb.csv", index=False)

df_all




BUILDING MB HISTOGRAMS FOR nominal
[OK] clone source mb_file0_fg2d_eemass_reco_nominal: TH2F  fg2d_eemass_reco_nominal_sim_icent0_file0
[OK] clone result mb_file0_fg2d_eemass_reco_nominal: TH2F  mb_file0_fg2d_eemass_reco_nominal
[OK] built mb_file0_fg2d_eemass_reco_nominal: integral=6.39584e+06
[OK] clone source mb_all_fg2d_eemass_reco_nominal: TH2F  mb_file0_fg2d_eemass_reco_nominal
[OK] clone result mb_all_fg2d_eemass_reco_nominal: TH2F  mb_all_fg2d_eemass_reco_nominal
[OK] built mb_all_fg2d_eemass_reco_nominal: integral=6.39584e+06
[OK] clone source mb_file0_fg2d_eemass_reco_nominal: TH2F  fg2d_eemass_reco_nominal_sim_icent0_file0
[OK] clone result mb_file0_fg2d_eemass_reco_nominal: TH2F  mb_file0_fg2d_eemass_reco_nominal
[OK] built mb_file0_fg2d_eemass_reco_nominal: integral=6.39584e+06
[OK] clone source mb_all_fg2d_eemass_reco_nominal: TH2F  mb_file0_fg2d_eemass_reco_nominal
[OK] clone result mb_all_fg2d_eemass_reco_nominal: TH2F  mb_all_fg2d_eemass_reco_nominal
[OK] built mb_a

,config,pt_min,pt_max,pt_center,y_ee,e_ee,y_eeg,e_eeg,raw_eeg,pol2_eeg,eeg_over_ee,ee_over_eeg,fit_mean,fit_sigma,fit_chi2,fit_ndf
0,nominal,0.6,0.795,0.6975,2.735023e+05,21385.169580,53765.114606,6412.171834,53765.114606,0.0,0.196580,5.086984,0.128633,0.008124,197.246822,53
1,nominal,0.8,0.995,0.8975,1.262033e+06,35527.045676,161955.037743,6299.675368,161955.037743,0.0,0.128329,7.792487,0.135019,0.009779,435.650309,86
2,nominal,1.0,1.195,1.0975,1.442338e+06,23408.039344,229347.220703,5096.491679,229347.220703,0.0,0.159011,6.288882,0.134339,0.012693,515.713866,86
3,nominal,1.2,1.395,1.2975,1.047900e+06,13895.137644,172909.202080,3280.953336,172909.202080,0.0,0.165005,6.060405,0.136687,0.012256,486.476898,86
4,nominal,1.4,1.595,1.4975,6.948852e+05,8072.020513,111093.359344,1955.499576,111093.359344,0.0,0.159873,6.254966,0.133760,0.012414,580.909491,86
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
191,escale101,3.0,3.495,3.2475,4.665921e+04,252.329471,12453.560440,100.045294,12453.560440,0.0,0.266905,3.746656,0.137949,0.017914,448.685156,86
192,escale101,3.5,3.995,3.7475,1.798468e+04,101.654028,4679.441997,40.356323,4679.441997,0.0,0.260190,3.843339,0.139270,0.018906,441.755114,86
193,escale101,4.0,4.995,4.4975,1.202511e+04,51.560706,3361.559862,22.338758,3361.559862,0.0,0.279545,3.577240,0.138993,0.020403,389.670966,86
194,escale101,5.0,6.995,5.9975,4.043003e+03,14.489883,1259.670494,7.132742,1259.670494,0.0,0.311568,3.209572,0.139630,0.022769,398.341401,86


In [51]:
df_all = add_ratio_columns(df_all)


In [52]:
c_ratio = root.TCanvas("c_ratio_nominal", "c_ratio_nominal", 800, 600)

g_ee_over_eeg = make_ratio_graph_from_df(
    df_all[df_all.config == "ecore05"],
    "ee_over_eeg",
    "ee_over_eeg_err",
    "g_ee_over_eeg_nominal",
    "nominal; p_{T} (GeV/c); ee / eeg"
)

c_ratio = root.TCanvas("c_ratio_nominal", "c_ratio_nominal", 600, 500)
g_ee_over_eeg.GetYaxis().SetRangeUser(0, 15)
g_ee_over_eeg.Draw("AP")

c_ratio.Draw()

In [53]:
import numpy as np

pt_orange = np.array([
    0.898, 1.105, 1.302, 1.504, 1.705, 1.905,
    2.253, 2.753, 3.251, 3.752, 4.500, 5.998, 8.496
])

eff_orange = np.array([
    0.0855, 0.0964, 0.1061, 0.1171, 0.1270, 0.1407,
    0.1563, 0.1836, 0.2112, 0.2335, 0.2668, 0.3169, 0.3744
])

ef_graph_wenqing = root.TGraph(len(pt_orange), array('d', pt_orange), array('d', eff_orange))

In [54]:
c12 = root.TCanvas("c_efficiency", "c_efficiency", 800, 600)
ef_graph_wenqing.SetMarkerStyle(20)
ef_graph_wenqing.SetMarkerColor(root.kBlue)
ef_graph_wenqing.SetLineColor(root.kBlue)
ef_graph_wenqing.Draw("AP")
c12.Draw()

In [55]:
import numpy as np

# Cu+Au @ 200 GeV, 0-20%
# ratio = gamma_hadron / gamma_pi0

pt_ratio = np.array([
    0.55, 0.60, 0.70, 0.80, 0.90, 1.00,
    1.25, 1.50, 1.75, 2.00, 2.25, 2.50,
    2.75, 3.00, 3.50, 4.00, 4.50, 5.00,
    5.50, 6.00, 6.50, 7.00, 7.50, 8.00,
    8.50, 9.00, 9.50, 9.90
])

ratio_vals = np.array([
    1.09696, 1.11315, 1.13620, 1.14979, 1.15773, 1.16320,
    1.17504, 1.18419, 1.19127, 1.19712, 1.20184, 1.20584,
    1.20926, 1.21221, 1.21708, 1.22092, 1.22365, 1.22579,
    1.22739, 1.22838, 1.22922, 1.22976, 1.22995, 1.23024,
    1.23024, 1.23028, 1.22996, 1.22980
])

def gamma_hadron_over_gamma_pi0(pt):
    return np.interp(pt, pt_ratio, ratio_vals)

In [56]:
#plotting r_gamma as g_ee_over_eeg * ef_graph_wenqing / gamma_hadron_over_gamma_pi0(pt) only for ecore05
c_final = root.TCanvas("c_final", "c_final", 800, 600)
g_final = root.TGraphErrors(len(df_all))
g_final.SetName("g_final")
g_final.SetTitle("Final; p_{T} (GeV/c); r_{#gamma}")
for i, row in df_all.iterrows():
    if row.config != "ecore05":
        continue
    pt = row.pt_center
    r_ee_eeg = row.ee_over_eeg
    eff = gamma_hadron_over_gamma_pi0(pt) / ef_graph_wenqing.Eval(pt)

    r_gamma = r_ee_eeg / eff

    g_final.SetPoint(i, pt, r_gamma)
    g_final.SetPointError(i, 0.5 * (row.pt_max - row.pt_min), 0.0)
g_final.SetMarkerStyle(20)
g_final.SetMarkerColor(root.kBlue)
g_final.SetLineColor(root.kBlue)
g_final.Draw("AP")
c_final.Draw()

In [57]:
ef_04_pt = [0.90,1.12,1.48,2.00,2.50,3.02,3.51,3.97,4.76]
ef_04_eps_t = [0.124,0.143,0.172,0.202,0.233,0.256,0.276,0.301,0.319]
ef_04_graph = root.TGraph(len(ef_04_pt), array('d', ef_04_pt), array('d', ef_04_eps_t))

import numpy as np

pt_orange = np.array([
    0.898, 1.105, 1.302, 1.504, 1.705, 1.905,
    2.253, 2.753, 3.251, 3.752, 4.500, 5.998, 8.496
])

eff_orange = np.array([
    0.0855, 0.0964, 0.1061, 0.1171, 0.1270, 0.1407,
    0.1563, 0.1836, 0.2112, 0.2335, 0.2668, 0.3169, 0.3744
])

graph_wenqing = root.TGraph(len(pt_orange), array('d', pt_orange), array('d', eff_orange))

In [58]:
#plotting r_gamma as g_ee_over_eeg * ef_graph_wenqing / gamma_hadron_over_gamma_pi0(pt)
c_final = root.TCanvas("c_final", "c_final", 800, 600)
g_final = root.TGraphErrors(len(df_all))
g_final.SetName("g_final")
g_final.SetTitle("Final; p_{T} (GeV/c); r_{#gamma}")
for i, row in df_all.iterrows():
    pt = row.pt_center
    r_ee_eeg = row.ee_over_eeg

    r_gamma = 1./r_ee_eeg

    g_final.SetPoint(i, pt, r_gamma)
    g_final.SetPointError(i, 0.5 * (row.pt_max - row.pt_min), 0.0)
g_final.SetMarkerStyle(20)
g_final.GetYaxis().SetRangeUser(0.0, 0.5)
g_final.GetXaxis().SetRangeUser(0., 4.0)
g_final.SetMarkerColor(root.kBlue)
g_final.SetLineColor(root.kBlue)
g_final.Draw("AP")
ef_04_graph.SetMarkerStyle(21)
ef_04_graph.SetMarkerColor(root.kRed)
ef_04_graph.SetLineColor(root.kRed)
ef_04_graph.Draw("P same")
graph_wenqing.SetMarkerStyle(22)
graph_wenqing.SetMarkerColor(root.kGreen+2)
graph_wenqing.SetLineColor(root.kGreen+2)
graph_wenqing.Draw("P same")
c_final.Draw()

In [59]:
def make_rgamma_graph_for_config(df, cfg, ef_graph, name_suffix=""):
    d = df[df["config"] == cfg].copy()
    d = d.sort_values("pt_center")

    g = root.TGraphErrors(len(d))
    g.SetName(f"g_rgamma_{cfg}{name_suffix}")
    g.SetTitle("Final; p_{T} (GeV/c); r_{#gamma}")

    for ip, row in enumerate(d.itertuples()):
        pt = row.pt_center

        r_ee_eeg = row.ee_over_eeg
        r_ee_eeg_err = getattr(row, "ee_over_eeg_err", 0.0)


        r_gamma = 1./  r_ee_eeg
        r_gamma_err = r_ee_eeg_err / r_ee_eeg**2 if r_ee_eeg != 0 else 0.0

        g.SetPoint(ip, pt, r_gamma)
        g.SetPointError(ip, 0.5 * (row.pt_max - row.pt_min), r_gamma_err)

    return g

In [60]:
configs_plot = [
    "nominal",
    "conv_solution",
    "conv_tight",
    "ecore03",
    "ecore05",
    "pte02",
    "pte04",
    "eid0",
    "eid1",
    "eid3",
    "chi2_4",
    "chi2_5",
    "escale099",
    "escale101",
]

colors = [
    root.kBlack,
    root.kBlue,
    root.kRed,
    root.kGreen + 2,
    root.kOrange + 7,
    root.kMagenta,
    root.kCyan + 2,
    root.kViolet,
    root.kAzure + 7,
    root.kPink + 7,
    root.kGray + 2,
    root.kSpring + 5,
    root.kOrange,
    root.kTeal + 3,
]

markers = [
    20, 21, 22, 23, 29, 33, 34,
    24, 25, 26, 27, 28, 30, 32
]

graphs_rgamma = {}

c_final = root.TCanvas("c_final_all_configs", "c_final_all_configs", 900, 700)

frame = root.TH2F(
    "frame_rgamma_all_configs",
    "Final; p_{T} (GeV/c); r_{#gamma}",
    100, 0.0, 4.0,
    100, 0.0, 0.5
)
frame.SetStats(0)
frame.Draw()

leg = root.TLegend(0.58, 0.16, 0.88, 0.55)
leg.SetBorderSize(0)
leg.SetFillStyle(0)
leg.SetTextSize(0.026)

for i, cfg in enumerate(configs_plot):
    g = make_rgamma_graph_for_config(df_all, cfg, ef_04_graph)

    g.SetMarkerStyle(markers[i])
    g.SetMarkerSize(1.1)
    if cfg == "pte02":
        g.SetMarkerSize(2.9)
    g.SetMarkerColor(colors[i])
    g.SetLineColor(colors[i])
    g.SetLineWidth(1)

    graphs_rgamma[cfg] = g

    draw_opt = "P same"
    g.Draw(draw_opt)

    leg.AddEntry(g, cfg, "p")

leg.Draw()

c_final.Draw()

In [61]:
print("printing from graphs_rgamma keys:", graphs_rgamma.keys())
g = graphs_rgamma["nominal"]
print("pt_nominal = [", ", ".join(f"{g.GetPointX(i):.2f}" for i in range(g.GetN())) + "]")
print("eps_f_nominal = [", ", ".join(f"{g.GetPointY(i):.2f}" for i in range(g.GetN())) + "]")
print("epsf_ecore03= [", ", ".join(f"{graphs_rgamma['ecore03'].GetPointY(i):.2f}" for i in range(graphs_rgamma['ecore03'].GetN())) + "]")
print("epsf_ecore05= [", ", ".join(f"{graphs_rgamma['ecore05'].GetPointY(i):.2f}" for i in range(graphs_rgamma['ecore05'].GetN())) + "]")


printing from graphs_rgamma keys: dict_keys(['nominal', 'conv_solution', 'conv_tight', 'ecore03', 'ecore05', 'pte02', 'pte04', 'eid0', 'eid1', 'eid3', 'chi2_4', 'chi2_5', 'escale099', 'escale101'])
pt_nominal = [ 0.70, 0.90, 1.10, 1.30, 1.50, 1.70, 1.90, 2.25, 2.75, 3.25, 3.75, 4.50, 6.00, 8.50]
eps_f_nominal = [ 0.20, 0.13, 0.16, 0.17, 0.16, 0.16, 0.18, 0.20, 0.22, 0.27, 0.26, 0.28, 0.31, 0.29]
epsf_ecore03= [ 0.23, 0.15, 0.19, 0.19, 0.21, 0.18, 0.22, 0.24, 0.26, 0.30, 0.30, 0.32, 0.35, 0.32]
epsf_ecore05= [ 0.12, 0.09, 0.13, 0.14, 0.13, 0.12, 0.14, 0.16, 0.17, 0.22, 0.22, 0.24, 0.27, 0.24]


In [62]:
import numpy as np

# Cu+Au @ 200 GeV, 0-20%
# ratio = gamma_hadron / gamma_pi0

pt_ratio = np.array([
    0.55, 0.60, 0.70, 0.80, 0.90, 1.00,
    1.25, 1.50, 1.75, 2.00, 2.25, 2.50,
    2.75, 3.00, 3.50, 4.00, 4.50, 5.00,
    5.50, 6.00, 6.50, 7.00, 7.50, 8.00,
    8.50, 9.00, 9.50, 9.90
])

ratio_vals = np.array([
    1.09696, 1.11315, 1.13620, 1.14979, 1.15773, 1.16320,
    1.17504, 1.18419, 1.19127, 1.19712, 1.20184, 1.20584,
    1.20926, 1.21221, 1.21708, 1.22092, 1.22365, 1.22579,
    1.22739, 1.22838, 1.22922, 1.22976, 1.22995, 1.23024,
    1.23024, 1.23028, 1.22996, 1.22980
])

def gamma_hadron_over_gamma_pi0(pt):
    return np.interp(pt, pt_ratio, ratio_vals)

In [63]:
embed = [ 0.713, 0.899, 0.937, 0.914, 0.881, 0.897, 0.886, 0.920, 0.919 ]
pt_embed = [ 1.5, 2.5, 3.5, 4.5, 5.5, 6.5, 7.5, 8.5, 9.5 ]
embed_graph = root.TGraph(len(pt_embed), array('d', pt_embed), array('d', embed))
print(embed_graph.Eval(3.5))

0.937


In [64]:
import numpy as np

pt_wenq = [0.9, 1.1, 1.3, 1.5, 1.7, 1.9, 2.25, 2.75, 3.25, 3.75, 4.5, 6.0, 8.5]
pt_err = [0.05] * len(pt_wenq)  # Assuming a constant error of 0.1 GeV/c for all points

rgamma_50_60 = [1.117, 1.145, 1.114, 1.144, 1.149, 1.138, 1.164, 1.159, 1.203, 1.182, 1.276, 1.202, 1.816]

stat_50_60 = [0.016, 0.019, 0.012, 0.013, 0.015, 0.017, 0.014, 0.022, 0.033, 0.047, 0.051, 0.072, 0.297]

syst_50_60 = [0.054, 0.055, 0.053, 0.055, 0.055, 0.054, 0.054, 0.053, 0.054, 0.052, 0.053, 0.050, 0.074]

rgamma_20_30 = [1.134, 1.141, 1.148, 1.146, 1.187, 1.205, 1.236, 1.243, 1.320, 1.350, 1.340, 1.813, 1.697]

stat_20_30 = [0.018, 0.014, 0.014, 0.014, 0.017, 0.019, 0.018, 0.024, 0.038, 0.036, 0.051, 0.118, 0.195]

syst_20_30 = [0.051, 0.051, 0.054, 0.052, 0.053, 0.054, 0.055, 0.054, 0.057, 0.057, 0.055, 0.073, 0.068]

rgamma_30_40 = [1.132, 1.140, 1.145, 1.151, 1.151, 1.195, 1.218, 1.217, 1.250, 1.286, 1.346, 1.436, 1.832]

stat_30_40 = [0.018, 0.014, 0.014, 0.015, 0.016, 0.019, 0.018, 0.023, 0.035, 0.032, 0.052, 0.082, 0.247]

syst_30_40 = [0.052, 0.052, 0.054, 0.053, 0.052, 0.054, 0.055, 0.054, 0.055, 0.055, 0.055, 0.058, 0.074]

graph_wenqing_rg = root.TGraphErrors(len(pt_wenq), array('d', pt_wenq), array('d', rgamma_30_40), array('d', pt_err), array('d', syst_30_40))


In [65]:
inc_over_tag_ratio = hists_read[0][0][0]  #Tgraph from file
epsilon_f = hists_read[1][0][0] 
rgamma_graph = root.TGraphErrors(len(pt_ratio))
for ipoint in range(inc_over_tag_ratio.GetN()):
    pt = inc_over_tag_ratio.GetX()[ipoint]
    ratio = inc_over_tag_ratio.GetY()[ipoint]
    ratio_err = inc_over_tag_ratio.GetEY()[ipoint]
    rgamma_val = 1.*ratio / gamma_hadron_over_gamma_pi0(pt) * epsilon_f.Eval(pt) * 0.9 #* embed_graph.Eval(pt)
    print(f"pt: {pt:.2f}, ratio: {ratio:.3f}, gamma_hadron_over_gamma_pi0: {gamma_hadron_over_gamma_pi0(pt):.3f}, embed: {embed_graph.Eval(pt):.3f}, epsilon_f: {epsilon_f.Eval(pt):.3f}, R_gamma: {rgamma_val:.3f}")
    rgamma_err= rgamma_val * ratio_err / ratio
    rgamma_graph.SetPoint(ipoint, pt, rgamma_val)
    rgamma_graph.SetPointError(ipoint, 0, rgamma_err)

c2 = root.TCanvas("c2","c2",1400,800)
rgamma_graph.SetMarkerStyle(20)
rgamma_graph.SetMarkerColor(root.kRed)
rgamma_graph.SetTitle("R_gamma vs pT; pT (GeV/c); R_gamma")
rgamma_graph.Draw("AP")
graph_wenqing_rg.SetMarkerStyle(20)
graph_wenqing_rg.SetMarkerColor(root.kBlue)
graph_wenqing_rg.Draw("P SAME")
c2.Draw()

NameError: name 'hists_read' is not defined

In [ ]:
pt_epsf_mb = [ 0.90, 1.25, 1.75, 2.25, 2.75, 3.25, 3.75, 4.50, 5.50, 6.50, 7.50, 8.50, 9.50 ]
epsf_mb = [ 0.101, 0.182, 0.212, 0.254, 0.278, 0.364, 0.340, 0.341, 0.410, 0.414, 0.409, 0.365, 0.386 ]

epsf_graph = root.TGraph(len(pt_epsf_mb), array('d', pt_epsf_mb), array('d', epsf_mb))
print(epsf_graph.Eval(0.9))

0.101
